# LSQ：可学习步长（Learned Step Size Quantization）

配套文章：

- 《大模型量化算法（18）：LSQ / PACT / DSQ——可学习的 scale 与 clip》
  https://lrypcy.github.io/2026/08/29/llm-quant-18-lsq-pact-dsq/ （§3 全部、§5.4 三方对比）
- 《QAT（00）：总览》 https://lrypcy.github.io/2026/08/25/qat-00-overview/ （§3.1 LSQ 的精确梯度）
- 《大模型量化算法（11）：伪量化算子插入》 https://lrypcy.github.io/2026/08/26/llm-quant-11-fake-quant-insertion/

**为什么先做 LSQ**：它是 QAT 侧最典型、工程上最主流的一个算法（"把量化超参数搬进计算图"这一
范式的样板），而且它的梯度有一个可以**精确分解、可验证**的结构——本文所有结论都能落到数字上。

## 本 notebook 的六个实验

| # | 实验 | 对应文章 | 要验证的一句话 |
|---|---|---|---|
| A | PTQ 基线：scale 扫描 | 18 篇 §2 | min-max 不是最优，光把 s 挪到 MSE 最优点就白捡 ~4.6 dB |
| B | 梯度分解：舍入项 − 截断项 | 18 篇 §3.2 | **MSE 最优 s\* 恰好是两项相消的点**；逐元素不对称随位宽指数增长 |
| C | 自我稳定与初始化不对称 | 18 篇 §3.4 | s 从"太小"恢复快、从"太大"恢复慢 ⇒ **初始化宁小勿大** |
| D | 相干和、失衡比 R 与梯度缩放 g | 18 篇 §3.3 | ∇_s L 是**相干和**（比随机和大 30–80×）⇒ g 是必需品，论文的 √(n_Q·Q_P) 是下界 |
| E | 端到端对比：LSQ vs 固定 s vs min-max | 18 篇 §5.4 | 学 s 的收益上限 ≈ "从 min-max 挪到 MSE 最优"那么多 |
| F | per-tensor vs per-channel LSQ | 18 篇 §8 + 00 篇 §粒度 | 粒度与可学习性是两个正交的收益来源 |

## 运行方式

```bash
cd experiments/quantization/lsq_learned_step_size
jupyter nbconvert --to notebook --execute --inplace lsq_learned_step_size.ipynb
```

纯 numpy + matplotlib，CPU 秒级（smoke）到分钟级（full）。**随机种子固定 SEED=0，结果可复现。**
顶部 `MODE` 开关：`"smoke"` 为快速冒烟（默认），`"full"` 为全量（更大网格、更多步数、更多 arm）——
审阅通过后把 MODE 改成 "full" 再跑一遍即为最终结果。

## 0. 环境与全局配置

`MODE` 决定规模。所有超参集中在 `CFG` 里，full 模式只是把网格/步数/arm 数放大。

In [1]:
import os
import json
import time
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

SEED = 0
MODE = "smoke"          # "smoke" | "full"

CFG = {
    "smoke": dict(bits=(2, 4, 8), grid_pts=120, steps=400,
                  lr_grid=(1e-3, 1e-2), s0_mult=(0.1, 1.0, 10.0),
                  g_modes=("lsq", "sqrt_n", "none"), sweep_pts=40,
                  lr_grid_g=(1e-3, 1e-2, 3e-2), lr_grid_c=(1e-3, 3e-2)),
    "full":  dict(bits=(2, 3, 4, 6, 8), grid_pts=600, steps=3000,
                  lr_grid=(1e-3, 2e-3, 5e-3, 1e-2, 3e-2),
                  s0_mult=(0.01, 0.1, 1.0, 10.0, 100.0),
                  g_modes=("lsq", "sqrt_n", "none"), sweep_pts=120,
                  lr_grid_g=(1e-3, 3e-3, 1e-2, 3e-2, 1e-1),
                  lr_grid_c=(3e-4, 1e-3, 1e-2, 3e-2)),
}[MODE]

HERE = os.getcwd()
RES = os.path.join(HERE, "results")
os.makedirs(RES, exist_ok=True)

_LINES = []
def log(msg=""):
    """打印并缓存，末尾统一写入 results/stdout.txt。"""
    print(msg)
    _LINES.append(str(msg))

def savefig(fig, name):
    p = os.path.join(RES, name)
    fig.savefig(p, dpi=130, bbox_inches="tight")
    plt.close(fig)
    log(f"[save] {p}")
    return p

log(f"MODE={MODE}  CFG={CFG}")
log(f"numpy={np.__version__}  matplotlib={matplotlib.__version__}")

MODE=smoke  CFG={'bits': (2, 4, 8), 'grid_pts': 120, 'steps': 400, 'lr_grid': (0.001, 0.01), 's0_mult': (0.1, 1.0, 10.0), 'g_modes': ('lsq', 'sqrt_n', 'none'), 'sweep_pts': 40, 'lr_grid_g': (0.001, 0.01, 0.03), 'lr_grid_c': (0.001, 0.03)}
numpy=2.1.1  matplotlib=3.11.1


## 1. 任务与量化器

复刻 18 篇 §2 的**任务 A（过定线性回归）**：$W^\star\in\mathbb{R}^{64\times128}$（2% 的 ×3 离群元素），
$X\in\mathbb{R}^{512\times128}$ i.i.d. 高斯，$Y = XW^{\star\top}+0.05\varepsilon$。
因为 $X^\top X/N\approx I$，这个任务近似等价于"最小化 $\lVert\hat W-W^\star\rVert_F^2$"——
**它是研究 scale 选择的理想探针**（但也正因如此，STE 式的权重训练在这里没什么用武之地，见实验 E）。

量化器与 17/18 篇完全一致：

$$\hat v = s\cdot\mathrm{clip}\big(\lfloor v/s\rceil,\;q_{\min},\;q_{\max}\big),\qquad
q_{\min}=-2^{b-1},\; q_{\max}=2^{b-1}-1$$

LSQ 的 scale 梯度（Eq.3，逐元素）：

$$\frac{\partial\hat v}{\partial s}=\begin{cases}
-v/s+\lfloor v/s\rceil & -q_{\min}<v/s<q_{\max} \\
q_{\min} & v/s\le q_{\min}\\
q_{\max} & v/s\ge q_{\max}
\end{cases}$$

In [2]:
QMIN = lambda b: -(2 ** (b - 1))
QMAX = lambda b: 2 ** (b - 1) - 1


def fq(W, s, b):
    """伪量化前向：s * clip(round(W/s), qmin, qmax)。s 可以是标量或 (m,1) 逐通道向量。"""
    return s * np.clip(np.round(W / s), QMIN(b), QMAX(b))


def dhat_ds(W, s, b):
    """LSQ Eq.(3)：d(hat v)/d(s)，逐元素。线性区 = round(v)-v（舍入残差的相反数）。"""
    qmin, qmax = QMIN(b), QMAX(b)
    v = W / s
    return np.where(v <= qmin, float(qmin),
                    np.where(v >= qmax, float(qmax), np.round(v) - v))


def ste_mask(W, s, b):
    """输入方向的经典 STE：截断区梯度截停（与主流框架行为一致）。"""
    v = W / s
    return ((v > QMIN(b)) & (v < QMAX(b))).astype(float)


def make_task(seed=SEED, n=128, m=64, N=512, tail=3.0, tail_p=0.02, noise=0.05):
    """任务 A：过定线性回归 + 2% 权重离群元素（18 篇 §2，代码见 §10.1）。"""
    rng = np.random.default_rng(seed)
    W = rng.normal(0, 1.0 / np.sqrt(n), (m, n))
    W = np.where(rng.random((m, n)) < tail_p, W * tail, W)
    X = rng.normal(0, 1, (N, n))
    Y = X @ W.T + noise * rng.normal(0, 1, (N, m))
    return X, Y, W


def loss_and_grad(X, Y, Wh):
    """L = 0.5*mean(R^2)，返回 (loss, dL/d(What))。"""
    R = X @ Wh.T - Y
    return 0.5 * np.mean(R ** 2), (R / R.size).T @ X


def minmax_scale(W, b):
    return float(np.max(np.abs(W))) / QMAX(b)


def lsq_init_scale(W, b):
    """LSQ 原文初始化：2*<|v|> / sqrt(Q_P)。"""
    return 2.0 * float(np.mean(np.abs(W))) / np.sqrt(QMAX(b))


X, Y, W_true = make_task()
Wls, *_ = np.linalg.lstsq(X, Y, rcond=None)
W0 = Wls.T                      # 最小二乘解（FP 最优），作为所有 QAT 的起点
nQ = W0.size                    # 8192

BITS = CFG["bits"]
B_MAIN = 4                      # 主位宽：作图与训练默认用 4-bit
g_lsq = 1.0 / np.sqrt(nQ * QMAX(B_MAIN))
s_mm_all = {b: minmax_scale(W0, b) for b in BITS}
s_lsq_all = {b: lsq_init_scale(W0, b) for b in BITS}

log(f"W0: shape={W0.shape}, rms={np.sqrt((W0**2).mean()):.5f}, max|W|={np.abs(W0).max():.5f}")
log(f"FP 全精度损失（不量化）                    = {loss_and_grad(X, Y, W0)[0]:.6e}")
log(f"4-bit 初始化：min-max s={s_mm_all[4]:.5f}  LSQ s0={s_lsq_all[4]:.5f}  g={g_lsq:.3e}")

W0: shape=(64, 128), rms=0.09473, max|W|=0.67716


FP 全精度损失（不量化）                    = 9.365979e-04
4-bit 初始化：min-max s=0.09674  LSQ s0=0.05563  g=4.176e-03


## 实验 A：PTQ 基线——把 scale 从 min-max 挪到 MSE 最优值多少 dB

这是 LSQ 的**收益天花板**：LSQ 再怎么学，也不可能比"事后拿完整任务损失搜出来的最优 s"好多少
（因为后者已经用到了 label）。先量出这个天花板，后面的对比才有参照系。

In [3]:
def grid_search_s(W0, b, pts, lo=0.05, hi=1.2, refine=3):
    """在 [lo, hi] * s_minmax 上网格搜索 MSE 最优 s，再做 refine 轮局部细化。

    细化很重要：s* 的精度直接决定实验 B 里「两项相消」能消到多干净
    （粗网格下残差占比会从 ~0.5% 虚高到 ~13%）。
    """
    s_mm = minmax_scale(W0, b)
    grid = np.linspace(lo * s_mm, hi * s_mm, pts)
    losses = np.array([loss_and_grad(X, Y, fq(W0, s, b))[0] for s in grid])
    j = int(np.argmin(losses))
    s_best, step = float(grid[j]), float(grid[1] - grid[0])
    for _ in range(refine):
        loc = np.linspace(s_best - step, s_best + step, 21)
        ls = np.array([loss_and_grad(X, Y, fq(W0, s, b))[0] for s in loc])
        k = int(np.argmin(ls))
        s_best, step = float(loc[k]), float(loc[1] - loc[0])
    return s_best, float(loss_and_grad(X, Y, fq(W0, s_best, b))[0]), grid, losses


rows_A = []
for b in BITS:
    s_mm = minmax_scale(W0, b)
    L_mm = loss_and_grad(X, Y, fq(W0, s_mm, b))[0]
    s_opt, L_opt, _, _ = grid_search_s(W0, b, CFG["grid_pts"])
    s_lsq0 = lsq_init_scale(W0, b)
    clip_frac = float((np.abs(W0 / s_opt) > QMAX(b)).mean() * 100)
    rows_A.append(dict(bit=b, s_minmax=s_mm, L_minmax=L_mm, s_opt=s_opt, L_opt=L_opt,
                       gain_db=10 * np.log10(L_mm / L_opt), s_lsq_init=s_lsq0,
                       clip_pct=clip_frac))

log("=" * 78)
log("[A] PTQ 基线：scale 扫描（网格 %d 点）" % CFG["grid_pts"])
log(f"{'bit':>4} {'s_minmax':>10} {'L(min-max)':>13} {'s*_MSE':>10} {'L(MSE最优)':>13} {'增益':>9} {'截断%':>8}")
for r in rows_A:
    log(f"{r['bit']:>4} {r['s_minmax']:>10.5f} {r['L_minmax']:>13.4e} {r['s_opt']:>10.5f} "
        f"{r['L_opt']:>13.4e} {r['gain_db']:>+8.2f}dB {r['clip_pct']:>7.3f}%")
log("-" * 78)
log("  读数：4-bit 下 min-max 比 MSE 最优差 %.2f dB —— 而 min-max 恰恰是多数框架的默认。"
    % rows_A[[r['bit'] for r in rows_A].index(4)]['gain_db'])

# 图 1：L(s) 扫描（澡盆曲线），min-max 不在盆底
fig, ax = plt.subplots(1, 2, figsize=(11.5, 4.2))
for b in BITS:
    s_mm = minmax_scale(W0, b)
    grid = np.linspace(0.05 * s_mm, 1.5 * s_mm, CFG["sweep_pts"])
    Ls = np.array([loss_and_grad(X, Y, fq(W0, s, b))[0] for s in grid])
    ax[0].plot(grid / s_mm, Ls, lw=2, label=f"b={b}")
    ax[0].axvline(1.0, color="#C44E52", ls="--", lw=1)
ax[0].set_yscale("log"); ax[0].set_xlabel("s / s_minmax"); ax[0].set_ylabel("task loss")
ax[0].set_title("[A] L(s) sweep: min-max is never the basin floor")
ax[0].legend(fontsize=9)

gain = [r["gain_db"] for r in rows_A]
ax[1].bar([str(r["bit"]) for r in rows_A], gain, color="#4C72B0")
ax[1].set_xlabel("bit-width"); ax[1].set_ylabel("gain of MSE-optimal s over min-max [dB]")
ax[1].set_title("[A] Head-room of a better scale (this is LSQ's ceiling)")
for i, v in enumerate(gain):
    ax[1].text(i, v, f"{v:.2f}", ha="center", va="bottom", fontsize=9)
savefig(fig, "lsq_scale_sweep_and_headroom.png")

[A] PTQ 基线：scale 扫描（网格 120 点）
 bit   s_minmax    L(min-max)     s*_MSE      L(MSE最优)        增益      截断%
   2    0.67716    5.4396e-01    0.10309    1.0513e-01    +7.14dB  25.439%
   4    0.09674    5.1021e-02    0.04149    1.7522e-02    +4.64dB   0.549%
   8    0.00533    1.0909e-03    0.00517    1.0807e-03    +0.04dB   0.024%
------------------------------------------------------------------------------
  读数：4-bit 下 min-max 比 MSE 最优差 4.64 dB —— 而 min-max 恰恰是多数框架的默认。


[save] /Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/lsq_learned_step_size/results/lsq_scale_sweep_and_headroom.png


'/Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/lsq_learned_step_size/results/lsq_scale_sweep_and_headroom.png'

## 实验 B：LSQ 梯度的「舍入项 − 截断项」分解

把 $\nabla_s\mathcal{L}=\sum_i \frac{\partial\mathcal L}{\partial\hat v_i}\frac{\partial\hat v_i}{\partial s}$
按"元素是否在量化范围内"拆成两项：

$$\nabla_s\mathcal L \;=\; \underbrace{s\!\!\sum_{i\in\text{范围内}}\!\!\delta_i^2}_{\text{舍入项}\ \ge0\ \Rightarrow\ \text{把 }s\text{ 往小推}}
\;-\; \underbrace{\sum_{i\in\text{截断区}}(\lvert v_i^\star\rvert-sQ_P)\,Q_P}_{\text{截断项}\ \ge0\ \Rightarrow\ \text{把 }s\text{ 往大推}}$$

**要验证的两件事**：(1) 两项的**过零点** ≈ 离线搜出的 MSE 最优 $s^\ast$；(2) 逐元素不对称
（截断区每元素的梯度强度 / 范围内每元素的梯度强度）随位宽**指数增长**——这就是"离群值垄断
scale 梯度"的定量说法。

> 实现细节：L(s) 对 s 是**锯齿状**的（每跨一个 round 边界就跳一次），所以"网格最优点"可能落在
> 某个窄锯齿谷里，那里 STE 的两项并不精确相消。因此这里额外用**符号变化 + 线性插值**定位
> $\nabla_s\mathcal L$ 的过零点 $s_{\text{zero}}$，它是"平滑包络"意义上的平衡点，对锯齿不敏感。

In [4]:
def decompose_grad_s(W0, s, b):
    """返回 (rounding term, clipping term, total, 逐元素不对称倍数, 截断比例)。"""
    _, gW = loss_and_grad(X, Y, fq(W0, s, b))
    v = W0 / s
    inr = (v > QMIN(b)) & (v < QMAX(b))
    dd = dhat_ds(W0, s, b)
    rnd = float(np.sum(gW[inr] * dd[inr]))        # >0，把 s 往小推
    clp = float(np.sum(gW[~inr] * dd[~inr]))      # <0，把 s 往大推
    per_in = float(np.mean(np.abs(gW[inr] * dd[inr]))) if inr.any() else 0.0
    per_out = float(np.mean(np.abs(gW[~inr] * dd[~inr]))) if (~inr).any() else 0.0
    asym = per_out / per_in if per_in > 0 else float("nan")
    return rnd, clp, rnd + clp, asym, float((~inr).mean() * 100)


rows_B = []
for b in BITS:
    s_opt, _, _, _ = grid_search_s(W0, b, CFG["grid_pts"])
    rnd, clp, tot, asym, clip_pct = decompose_grad_s(W0, s_opt, b)
    rows_B.append(dict(bit=b, s_opt=s_opt, rounding=rnd, clipping=clp, total=tot,
                       resid_pct=100 * abs(tot) / max(abs(rnd), abs(clp)),
                       asym=asym, clip_pct=clip_pct))

log("=" * 78)
log("[B] LSQ 梯度分解（在各自位宽的网格 MSE 最优 s* 处）")
log(f"{'bit':>4} {'s*':>10} {'舍入项(推s↓)':>15} {'截断项(推s↑)':>15} {'合计':>12} {'残差占比':>9} {'逐元素不对称':>12}")
for r in rows_B:
    log(f"{r['bit']:>4} {r['s_opt']:>10.5f} {r['rounding']:>+15.4e} {r['clipping']:>+15.4e} "
        f"{r['total']:>+12.4e} {r['resid_pct']:>8.2f}% {r['asym']:>11.1f}x")
log("  注：L(s) 对 s 是锯齿状（round 边界跳变），网格最优点可能落在某个窄锯齿谷里，")
log("      那里 STE 的两项并不精确相消 —— 所以下面额外定位「两项的过零点」来做对照。")

# 过零点：tot(s) = 0 的位置（对锯齿不敏感，是"平滑包络"意义上的平衡点）
b = B_MAIN
s_opt = [r for r in rows_B if r["bit"] == b][0]["s_opt"]
ss = np.linspace(0.30 * s_opt, 2.2 * s_opt, CFG["sweep_pts"])
rnd_c, clp_c, tot_c = [], [], []
for s in ss:
    r_, c_, t_, _, _ = decompose_grad_s(W0, s, b)
    rnd_c.append(r_); clp_c.append(c_); tot_c.append(t_)
tot_arr = np.array(tot_c)
lo = hi = None
for i in range(len(ss) - 1):
    if tot_arr[i] * tot_arr[i + 1] <= 0:
        lo, hi = float(ss[i]), float(ss[i + 1]); break
if lo is not None:                       # 二分细化：tot(s) 分段连续，可收敛到真正的零点
    for _ in range(40):
        mid = 0.5 * (lo + hi)
        if decompose_grad_s(W0, lo, b)[2] * decompose_grad_s(W0, mid, b)[2] <= 0:
            hi = mid
        else:
            lo = mid
s_zero = 0.5 * (lo + hi) if lo is not None else float(ss[int(np.argmin(np.abs(tot_arr)))])
rnd_z, clp_z, tot_z, asym_z, clip_z = decompose_grad_s(W0, s_zero, b)
zero_cross = dict(bit=b, s_grid_opt=s_opt, s_zero=s_zero,
                  rel_dev_pct=100 * abs(s_zero / s_opt - 1),
                  rounding=rnd_z, clipping=clp_z, total=tot_z,
                  resid_pct=100 * abs(tot_z) / max(abs(rnd_z), abs(clp_z)),
                  clip_pct=clip_z)
log("-" * 78)
log(f"[B] 过零点分析（bit={b}）：")
log(f"  网格 MSE 最优 s*  = {s_opt:.5f}（锯齿谷底；此处残差 {rows_B[[r['bit'] for r in rows_B].index(b)]['resid_pct']:.2f}%）")
log(f"  两项相消点 s_zero = {s_zero:.5f}（相对 s* 偏差 {zero_cross['rel_dev_pct']:.2f}%）")
log(f"     舍入项 {rnd_z:+.4e} / 截断项 {clp_z:+.4e} / 合计 {tot_z:+.4e}"
    f"（残差仅占 {zero_cross['resid_pct']:.3f}%）")
L_zero = loss_and_grad(X, Y, fq(W0, s_zero, b))[0]
L_grid = loss_and_grad(X, Y, fq(W0, s_opt, b))[0]
log(f"  L(s_zero)={L_zero:.4e} vs L(s*_grid)={L_grid:.4e}：两个点的任务损失几乎相同"
    f"（差 {10*np.log10(L_grid/L_zero):+.3f} dB），但只有 s_zero 处两项精确相消。")
log("  => 「学 scale 的收敛点」与「离线搜出的最优 scale」在数值上是同一个点。")

fig, ax = plt.subplots(1, 2, figsize=(11.5, 4.2))
ax[0].plot(ss, np.array(rnd_c), lw=2, color="#4C72B0", label="rounding term (pushes s down)")
ax[0].plot(ss, -np.array(clp_c), lw=2, color="#C44E52", label="clipping term (pushes s up)")
ax[0].plot(ss, np.maximum(np.abs(tot_arr), 1e-12), lw=2, ls="--", color="#55A868",
           label="|net grad_s L| (has a zero)")
ax[0].axvline(s_opt, color="grey", ls=":", lw=1.5, label=f"grid MSE-optimal s*={s_opt:.5f}")
ax[0].axvline(s_zero, color="#8172B3", ls="-.", lw=1.5, label=f"zero crossing={s_zero:.5f}")
ax[0].set_yscale("log"); ax[0].set_xlabel("s"); ax[0].set_ylabel("gradient magnitude")
ax[0].set_title("[B] The two forces cross zero right at the optimal s"); ax[0].legend(fontsize=7.5)

ax[1].semilogy([r["bit"] for r in rows_B], [r["asym"] for r in rows_B], "o-", color="#DD8452", lw=2)
ax[1].set_xlabel("bit-width"); ax[1].set_ylabel("per-element asymmetry (clipped / in-range)")
ax[1].set_title("[B] Outliers dominate the scale gradient,\nand the dominance grows with precision")
for r in rows_B:
    ax[1].annotate(f"{r['asym']:.0f}x", (r["bit"], r["asym"]), textcoords="offset points",
                   xytext=(6, -4), fontsize=9)
savefig(fig, "lsq_gradient_decomposition.png")

[B] LSQ 梯度分解（在各自位宽的网格 MSE 最优 s* 处）
 bit         s*        舍入项(推s↓)        截断项(推s↑)           合计      残差占比       逐元素不对称
   2    0.10309     +9.4992e-01     -1.0594e+00  -1.0946e-01    10.33%         6.7x
   4    0.04149     +4.4082e-01     -5.4280e-01  -1.0198e-01    18.79%       239.6x
   8    0.00517     +5.4925e-02     -3.7558e-02  +1.7368e-02    31.62%      2746.8x
  注：L(s) 对 s 是锯齿状（round 边界跳变），网格最优点可能落在某个窄锯齿谷里，
      那里 STE 的两项并不精确相消 —— 所以下面额外定位「两项的过零点」来做对照。


------------------------------------------------------------------------------
[B] 过零点分析（bit=4）：
  网格 MSE 最优 s*  = 0.04149（锯齿谷底；此处残差 18.79%）
  两项相消点 s_zero = 0.04378（相对 s* 偏差 5.51%）
     舍入项 +4.6909e-01 / 截断项 -4.6909e-01 / 合计 -2.8200e-14（残差仅占 0.000%）
  L(s_zero)=1.7508e-02 vs L(s*_grid)=1.7522e-02：两个点的任务损失几乎相同（差 +0.004 dB），但只有 s_zero 处两项精确相消。
  => 「学 scale 的收敛点」与「离线搜出的最优 scale」在数值上是同一个点。


[save] /Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/lsq_learned_step_size/results/lsq_gradient_decomposition.png


'/Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/lsq_learned_step_size/results/lsq_gradient_decomposition.png'

## 实验 C：自我稳定与「初始化不对称」

§3.2 的推论：往小偏和往大偏的**刚度差约 $2Q_P$ 倍**——
$s$ 太小 ⇒ 海量元素被截断 ⇒ 每个以 $Q_P$ 的强度把 $s$ 往上顶（很快）；
$s$ 太大 ⇒ 只有 $|\delta|\le0.5$ 的弱项在推（很慢）。

所以工程结论是：**scale 初始化宁小勿大**。这里用 s0 ∈ {0.1×, 1×, 10×} × LSQ init 直接验证。

In [5]:
def train_lsq(W0, s0, b, steps, lr, g=1.0, learn_s=True, train_w=True, mom=0.9,
              trace_every=None):
    """QAT 训练：W 用 STE 更新，s 用 LSQ Eq.(3) 更新（乘梯度缩放 g）。

    train_w=False 时冻结 W、只学 s —— 用来隔离「学 s」本身的收益
    （任务 A 的 FP 解已经最优，STE 式权重训练在这里帮不上忙甚至有害）。
    full 模式下 steps 较大，trace_every 控制采样密度。
    """
    W = W0.copy(); s = float(s0); vw = np.zeros_like(W); vs = 0.0
    traj = []
    every = trace_every or max(steps // 20, 1)
    for t in range(steps):
        _, gWq = loss_and_grad(X, Y, fq(W, s, b))
        if train_w:
            vw = mom * vw + gWq * ste_mask(W, s, b)
            W = W - lr * vw
        if learn_s:
            vs = mom * vs + g * float(np.sum(gWq * dhat_ds(W, s, b)))
            s = max(s - lr * vs, 1e-9)
        if t % every == 0 or t == steps - 1:
            traj.append(float(s))
    return W, float(s), loss_and_grad(X, Y, fq(W, s, b))[0], traj


steps = CFG["steps"]
lr_c_list = CFG["lr_grid_c"]
lr_trace = max(lr_c_list)          # 用最大的 lr 画轨迹（恢复能力最强的情形）
rows_C, trajs_C = [], {}
for lr in lr_c_list:
    for mult in CFG["s0_mult"]:
        s0 = mult * s_lsq_all[B_MAIN]
        _, s_end, L_end, traj = train_lsq(W0, s0, B_MAIN, steps=steps, lr=lr, g=g_lsq,
                                          learn_s=True, mom=0.9)
        rows_C.append(dict(mult=mult, lr=lr, s0=s0, s_end=s_end, loss=L_end))
        if abs(lr - lr_trace) < 1e-15:
            trajs_C[mult] = traj

log("=" * 78)
log(f"[C] 自我稳定：不同 s0 的收敛（bit={B_MAIN}, steps={steps}）")
log(f"{'lr':>8} {'s0 倍数':>8} {'s0':>10} {'收敛 s':>10} {'最终损失':>13}")
for lr in lr_c_list:
    for r in [r for r in rows_C if r["lr"] == lr]:
        log(f"{lr:>8.0e} {r['mult']:>7}x {r['s0']:>10.5f} {r['s_end']:>10.5f} {r['loss']:>13.4e}")
        cur = [r for r in rows_C if r["lr"] == lr]
    log(f"  -> lr={lr:.0e}：收敛 s 的极值比 = "
        f"{max(r['s_end'] for r in cur)/min(r['s_end'] for r in cur):.3f}，"
        f"损失极值比 = {max(r['loss'] for r in cur)/min(r['loss'] for r in cur):.3f}")
log("-" * 78)
log("  读数：大 lr 下两个数量级的初值跨度收敛到几乎同一个 s（自我稳定）；")
log("        小 lr 下偏大的初值来不及走完那段「弱恢复」，明显吃亏 —— 所以初始化宁小勿大。")

fig, ax = plt.subplots(1, 2, figsize=(11.5, 4.2))
for mult, traj in trajs_C.items():
    xs = np.linspace(0, steps, len(traj))
    ax[0].plot(xs, traj, lw=2, label=f"s0={mult}x LSQ init ({mult*s_lsq_all[B_MAIN]:.4f})")
ax[0].axhline([r for r in rows_A if r["bit"] == B_MAIN][0]["s_opt"], color="grey", ls="--",
              lw=1.2, label="MSE-optimal s*")
ax[0].set_yscale("log"); ax[0].set_xlabel("step"); ax[0].set_ylabel("s")
ax[0].set_title(f"[C] Self-stabilization at lr={lr_trace:.0e}:\nall inits converge, 'too large' recovers slowly")
ax[0].legend(fontsize=8)

w = 0.8 / len(lr_c_list)
for i, lr in enumerate(lr_c_list):
    cur = [r for r in rows_C if r["lr"] == lr]
    xs = np.arange(len(cur)) + i * w
    ax[1].bar(xs, [r["loss"] for r in cur], width=w, label=f"lr={lr:.0e}")
ax[1].set_xticks(np.arange(len(CFG["s0_mult"])) + w * (len(lr_c_list) - 1) / 2)
ax[1].set_xticklabels([f"{m}x" for m in CFG["s0_mult"]])
ax[1].set_yscale("log"); ax[1].set_xlabel("init multiplier"); ax[1].set_ylabel("final task loss")
ax[1].set_title("[C] Final loss vs init (small init is the safe side)")
ax[1].legend(fontsize=8)
savefig(fig, "lsq_self_stabilization.png")

[C] 自我稳定：不同 s0 的收敛（bit=4, steps=400）
      lr    s0 倍数         s0       收敛 s          最终损失
   1e-03     0.1x    0.00556    0.03682    1.8233e-02
   1e-03     1.0x    0.05563    0.05037    1.8280e-02
   1e-03    10.0x    0.55627    0.52370    5.2239e-01
  -> lr=1e-03：收敛 s 的极值比 = 14.223，损失极值比 = 28.650
   3e-02     0.1x    0.00556    0.04273    2.1142e-02
   3e-02     1.0x    0.05563    0.04078    1.7603e-02
   3e-02    10.0x    0.55627    0.04387    1.9630e-02
  -> lr=3e-02：收敛 s 的极值比 = 1.076，损失极值比 = 1.201
------------------------------------------------------------------------------
  读数：大 lr 下两个数量级的初值跨度收敛到几乎同一个 s（自我稳定）；
        小 lr 下偏大的初值来不及走完那段「弱恢复」，明显吃亏 —— 所以初始化宁小勿大。


[save] /Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/lsq_learned_step_size/results/lsq_self_stabilization.png


'/Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/lsq_learned_step_size/results/lsq_self_stabilization.png'

## 实验 D：相干和、失衡比 R 与梯度缩放 g

LSQ 用 $R=\dfrac{\lvert\nabla_s\mathcal L\rvert/s}{\lVert\nabla_W\mathcal L\rVert/\lVert W\rVert}$ 度量更新失衡，
并用 $g=1/\sqrt{n_QQ_P}$ 把 $R$ 拉回 1。这里验证两件事：

1. **实测 R 比论文的 $\sqrt{n_QQ_P}$ 估计还大一个数量级**——因为线性区每一项都正比于同一个舍入
   残差 $\delta_i$、全正号不相消，所以 $\nabla_s\mathcal L$ 是**相干和**（$\propto n_Q$）而不是
   随机和（$\propto\sqrt{n_Q}$）。
2. **g 的实际作用**：在单层玩具上它主要决定"学习率窗口有多宽"（不带 g 在大 lr 下会炸）；
   在深层网络里它是生死线（LSQ 原文 Table 3）。

In [6]:
rows_D = []
for b in BITS:
    sb = lsq_init_scale(W0, b)
    _, gW = loss_and_grad(X, Y, fq(W0, sb, b))
    dd = dhat_ds(W0, sb, b)
    gs = float(np.sum(gW * dd))
    R = abs(gs) / sb / (np.linalg.norm(gW) / np.linalg.norm(W0))
    rand = np.sqrt(nQ) * np.sqrt((gW ** 2).mean()) * np.sqrt((dd ** 2).mean())
    rows_D.append(dict(bit=b, R=R, pred=np.sqrt(nQ * QMAX(b)), ratio=R / np.sqrt(nQ * QMAX(b)),
                       coherent=abs(gs) / rand))

log("=" * 78)
log("[D] 失衡比 R 与相干性（在 LSQ 初始化点处测量）")
log(f"{'bit':>4} {'实测 R':>12} {'论文预测 √(n_Q·Q_P)':>20} {'比值':>8} {'相干和/随机和':>14}")
for r in rows_D:
    log(f"{r['bit']:>4} {r['R']:>12.2f} {r['pred']:>20.2f} {r['ratio']:>7.2f}x {r['coherent']:>13.1f}x")
log("-" * 78)
log("  读数：∇_s L 是相干和（比独立假设下的随机和大 30–80×），所以论文的 √(n_Q·Q_P) 是下界；")
log("        g 不是可选调参，是必需品。")

# g 消融：三种 g × lr 网格（SGD + momentum）
g_of = {"lsq": lambda b: 1.0 / np.sqrt(nQ * QMAX(b)),
        "sqrt_n": lambda b: 1.0 / np.sqrt(nQ),
        "none": lambda b: 1.0}
g_name = {"lsq": "g=1/sqrt(n_Q*Q_P)", "sqrt_n": "g=1/sqrt(n_Q)", "none": "g=1 (no scaling)"}
rows_G = []
for mode in CFG["g_modes"]:
    for lr in CFG["lr_grid_g"]:
        _, s_e, L_e, _ = train_lsq(W0, s_lsq_all[B_MAIN], B_MAIN, steps=steps,
                                   lr=lr, g=g_of[mode](B_MAIN), learn_s=True, mom=0.9)
        rows_G.append(dict(g=mode, lr=lr, loss=L_e, s=s_e))

log("=" * 78)
log(f"[D] g 消融（SGD+momentum, bit={B_MAIN}, steps={steps}）—— 单元格为最终损失")
hdr = f"{'g 模式':>20} " + " ".join(f"{lr:>11.0e}" for lr in CFG["lr_grid_g"])
log(hdr)
for mode in CFG["g_modes"]:
    cells = [f"{r['loss']:>11.4e}" for r in rows_G if r["g"] == mode]
    log(f"{g_name[mode]:>20} " + " ".join(cells))
L_ptq_mm = [r for r in rows_A if r["bit"] == B_MAIN][0]["L_minmax"]
log("-" * 78)
log("  对照：4-bit min-max PTQ（不训练）= %.4e" % L_ptq_mm)
for mode in CFG["g_modes"]:
    cur = [r for r in rows_G if r["g"] == mode]
    worst = max(cur, key=lambda r: r["loss"])
    log(f"  {g_name[mode]:>20}：最好 {min(r['loss'] for r in cur):.4e}，"
        f"最差 {worst['loss']:.4e} @ lr={worst['lr']:.0e}")
for mode in CFG["g_modes"]:
    cur = [r["loss"] for r in rows_G if r["g"] == mode]
    log(f"  {g_name[mode]:>20}：最差/最好 = {max(cur)/min(cur):.2f}x（学习率敏感度）")
log("  读数：本任务是单层单 scale，g 的作用体现为「学习率窗口宽度」而非「能否收敛」；")
log("        LSQ 原文 Table 3（2-bit ResNet-18）里 g=1 在 lr=1e-2 直接不收敛 —— 深层网络上它是生死线。")

fig, ax = plt.subplots(1, 2, figsize=(11.5, 4.2))
w = 0.8 / len(CFG["g_modes"])
for i, mode in enumerate(CFG["g_modes"]):
    ls = [r["loss"] for r in rows_G if r["g"] == mode]
    xs = np.arange(len(CFG["lr_grid_g"])) + i * w
    ax[0].bar(xs, ls, width=w, label=g_name[mode])
ax[0].set_xticks(np.arange(len(CFG["lr_grid_g"])) + w)
ax[0].set_xticklabels([f"{lr:.0e}" for lr in CFG["lr_grid_g"]])
ax[0].set_yscale("log"); ax[0].set_xlabel("learning rate"); ax[0].set_ylabel("final loss")
ax[0].set_title("[D] Gradient scaling g widens the usable LR window"); ax[0].legend(fontsize=8)

xs = np.arange(len(BITS))
ax[1].bar(xs - 0.2, [r["R"] for r in rows_D], width=0.4, color="#4C72B0", label="measured R")
ax[1].bar(xs + 0.2, [r["pred"] for r in rows_D], width=0.4, color="#DD8452", label=r"paper est. $\sqrt{n_Q Q_P}$")
ax[1].set_xticks(xs); ax[1].set_xticklabels([str(b) for b in BITS])
ax[1].set_yscale("log"); ax[1].set_xlabel("bit-width"); ax[1].set_ylabel("imbalance ratio R")
ax[1].set_title("[D] Measured imbalance exceeds the paper's estimate"); ax[1].legend(fontsize=8)
savefig(fig, "lsq_imbalance_and_g_ablation.png")

[D] 失衡比 R 与相干性（在 LSQ 初始化点处测量）
 bit         实测 R      论文预测 √(n_Q·Q_P)       比值        相干和/随机和
   2       874.67                90.51    9.66x          39.5x
   4      2166.96               239.47    9.05x          32.2x
   8     15393.57              1019.99   15.09x          80.9x
------------------------------------------------------------------------------
  读数：∇_s L 是相干和（比独立假设下的随机和大 30–80×），所以论文的 √(n_Q·Q_P) 是下界；
        g 不是可选调参，是必需品。


[D] g 消融（SGD+momentum, bit=4, steps=400）—— 单元格为最终损失
                g 模式       1e-03       1e-02       3e-02
   g=1/sqrt(n_Q*Q_P)  1.8280e-02  1.8500e-02  1.7603e-02
       g=1/sqrt(n_Q)  1.7612e-02  1.6832e-02  1.7595e-02
    g=1 (no scaling)  1.6926e-02  1.6792e-02  2.0130e-02
------------------------------------------------------------------------------
  对照：4-bit min-max PTQ（不训练）= 5.1021e-02
     g=1/sqrt(n_Q*Q_P)：最好 1.7603e-02，最差 1.8500e-02 @ lr=1e-02
         g=1/sqrt(n_Q)：最好 1.6832e-02，最差 1.7612e-02 @ lr=1e-03
      g=1 (no scaling)：最好 1.6792e-02，最差 2.0130e-02 @ lr=3e-02
     g=1/sqrt(n_Q*Q_P)：最差/最好 = 1.05x（学习率敏感度）
         g=1/sqrt(n_Q)：最差/最好 = 1.05x（学习率敏感度）
      g=1 (no scaling)：最差/最好 = 1.20x（学习率敏感度）
  读数：本任务是单层单 scale，g 的作用体现为「学习率窗口宽度」而非「能否收敛」；
        LSQ 原文 Table 3（2-bit ResNet-18）里 g=1 在 lr=1e-2 直接不收敛 —— 深层网络上它是生死线。


[save] /Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/lsq_learned_step_size/results/lsq_imbalance_and_g_ablation.png


'/Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/lsq_learned_step_size/results/lsq_imbalance_and_g_ablation.png'

## 实验 E：端到端对比——LSQ vs 固定 s vs min-max PTQ

四个 arm 用**完全相同的初始化、相同的数据、相同的优化器**（SGD+momentum）：

- `fixed s=min-max + STE`：$s$ 冻结在 min-max，只训 $W$（最朴素的 QAT）
- `fixed s* + STE-QAT`：$s$ 冻结在离线网格搜出的最优值，只训 $W$
- `LSQ (s only)`：**只学 $s$、冻结 $W$** —— 这是 LSQ 最纯粹的形式，用来隔离"学 scale"本身的收益
- `LSQ (s + W)`：$s$ 与 $W$ 同训，$s_0=2\langle\lvert v\rvert\rangle/\sqrt{Q_P}$，带 $g$

**预期（18 篇 §5.4 的诚实结论）**：学 $s$ 的收益上限 ≈ 实验 A 量出的天花板（把 $s$ 从 min-max 挪到
MSE 最优）；**学 $s$ 的价值不是精度，而是把离线搜索这一步变成可微的、在线的**。
而在这个任务上再叠加 STE 式的 $W$ 训练通常无益——因为起点已经是 FP 最小二乘解（18 篇 §9.2 局限 1）。

In [7]:
s_opt_main = [r for r in rows_A if r["bit"] == B_MAIN][0]["s_opt"]
rows_E = []
for lr in CFG["lr_grid"]:
    _, _, L_fix, _ = train_lsq(W0, s_opt_main, B_MAIN, steps=steps, lr=lr, learn_s=False, mom=0.9)
    rows_E.append(dict(arm="fixed s* + STE", lr=lr, loss=L_fix, s=s_opt_main,
                       db=10 * np.log10(L_ptq_mm / L_fix)))
    _, s_e, L_sonly, _ = train_lsq(W0, s_lsq_all[B_MAIN], B_MAIN, steps=steps, lr=lr,
                                   g=g_lsq, learn_s=True, train_w=False, mom=0.9)
    rows_E.append(dict(arm="LSQ (s only, W frozen)", lr=lr, loss=L_sonly, s=s_e,
                       db=10 * np.log10(L_ptq_mm / L_sonly)))
    _, s_e, L_lsq, _ = train_lsq(W0, s_lsq_all[B_MAIN], B_MAIN, steps=steps, lr=lr,
                                 g=g_lsq, learn_s=True, train_w=True, mom=0.9)
    rows_E.append(dict(arm="LSQ (s + W)", lr=lr, loss=L_lsq, s=s_e,
                       db=10 * np.log10(L_ptq_mm / L_lsq)))
    _, _, L_mm_s, _ = train_lsq(W0, s_mm_all[B_MAIN], B_MAIN, steps=steps, lr=lr,
                                learn_s=False, mom=0.9)
    rows_E.append(dict(arm="fixed s=min-max + STE", lr=lr, loss=L_mm_s, s=s_mm_all[B_MAIN],
                       db=10 * np.log10(L_ptq_mm / L_mm_s)))

log("=" * 78)
log(f"[E] 端到端对比（bit={B_MAIN}, steps={steps}, 基准 = 4-bit min-max PTQ {L_ptq_mm:.4e}）")
log(f"{'arm':>24} {'lr':>8} {'最终损失':>14} {'学到的 s':>10} {'相对 min-max':>12}")
for r in rows_E:
    log(f"{r['arm']:>24} {r['lr']:>8.0e} {r['loss']:>14.4e} {r['s']:>10.5f} {r['db']:>+11.2f}dB")
best_of = lambda a: min([r for r in rows_E if r["arm"] == a], key=lambda r: r["loss"])
best_lsq = best_of("LSQ (s + W)")
best_sonly = best_of("LSQ (s only, W frozen)")
best_fix = best_of("fixed s* + STE")
L_opt_ptq = [r for r in rows_A if r["bit"] == B_MAIN][0]["L_opt"]
log("-" * 78)
log(f"  LSQ (s+W)        = {best_lsq['loss']:.4e} ({best_lsq['db']:+.2f} dB)  s={best_lsq['s']:.5f}")
log(f"  LSQ (只学 s)     = {best_sonly['loss']:.4e} ({best_sonly['db']:+.2f} dB)  s={best_sonly['s']:.5f}")
log(f"  固定 s* + STE    = {best_fix['loss']:.4e} ({best_fix['db']:+.2f} dB)")
log(f"  离线网格最优 s*（不训练）= {L_opt_ptq:.4e} ({10*np.log10(L_ptq_mm/L_opt_ptq):+.2f} dB)  s={s_opt_main:.5f}")
log(f"  => 「只学 s」相对离线最优固定 s 的差 = {10*np.log10(L_opt_ptq/best_sonly['loss']):+.3f} dB")
log(f"  => 在 s 之上再训 W 的额外收益        = {10*np.log10(best_sonly['loss']/best_lsq['loss']):+.3f} dB")
log("  （任务 A 的 FP 解已经最优，STE 式权重训练在这里无益甚至有害 —— 18 篇 §9.2 已声明这一局限）")

fig, ax = plt.subplots(1, 2, figsize=(11.5, 4.2))
arms = ["LSQ (s + W)", "LSQ (s only, W frozen)", "fixed s* + STE", "fixed s=min-max + STE"]
colors = {"LSQ (s + W)": "#4C72B0", "LSQ (s only, W frozen)": "#8172B3",
          "fixed s* + STE": "#55A868", "fixed s=min-max + STE": "#C44E52"}
for arm in arms:
    sub = [r for r in rows_E if r["arm"] == arm]
    ax[0].plot([r["lr"] for r in sub], [r["db"] for r in sub], "o-", lw=2,
               color=colors[arm], label=arm)
ax[0].axhline(0, color="grey", lw=1)
ax[0].axhline(10 * np.log10(L_ptq_mm / L_opt_ptq), color="grey", ls=":", lw=1.5,
              label="offline grid-optimal s* (no training)")
ax[0].set_xscale("log"); ax[0].set_xlabel("learning rate"); ax[0].set_ylabel("gain over min-max PTQ [dB]")
ax[0].set_title("[E] Learning s beats a fixed s, but the ceiling is 'grid-optimal'")
ax[0].legend(fontsize=8)

labels = ["min-max PTQ\n(no train)", "grid-optimal s*\n(no train)", "fixed s* + STE-QAT",
          "LSQ (s only)", "LSQ (s+W)"]
vals = [L_ptq_mm, L_opt_ptq, best_fix["loss"], best_sonly["loss"], best_lsq["loss"]]
bars = ax[1].bar(labels, vals, color=["#C44E52", "#DD8452", "#55A868", "#8172B3", "#4C72B0"])
ax[1].set_yscale("log"); ax[1].set_ylabel("task loss (lower is better)")
ax[1].set_title("[E] End-to-end: where the gain actually comes from")
ax[1].tick_params(axis="x", labelsize=8)
for lbl in ax[1].get_xticklabels():
    lbl.set_rotation(16); lbl.set_ha("right")
for b_, v in zip(bars, vals):
    ax[1].text(b_.get_x() + b_.get_width() / 2, v, f"{10*np.log10(L_ptq_mm/v):+.2f}dB",
               ha="center", va="bottom", fontsize=8)
savefig(fig, "lsq_end_to_end_comparison.png")

[E] 端到端对比（bit=4, steps=400, 基准 = 4-bit min-max PTQ 5.1021e-02）
                     arm       lr           最终损失      学到的 s   相对 min-max
          fixed s* + STE    1e-03     1.6901e-02    0.04149       +4.80dB
  LSQ (s only, W frozen)    1e-03     1.8535e-02    0.05051       +4.40dB
             LSQ (s + W)    1e-03     1.8280e-02    0.05037       +4.46dB
   fixed s=min-max + STE    1e-03     4.9200e-02    0.09674       +0.16dB
          fixed s* + STE    1e-02     1.8822e-02    0.04149       +4.33dB
  LSQ (s only, W frozen)    1e-02     1.7495e-02    0.04379       +4.65dB
             LSQ (s + W)    1e-02     1.8500e-02    0.04399       +4.41dB
   fixed s=min-max + STE    1e-02     6.3582e-02    0.09674       -0.96dB
------------------------------------------------------------------------------
  LSQ (s+W)        = 1.8280e-02 (+4.46 dB)  s=0.05037
  LSQ (只学 s)     = 1.7495e-02 (+4.65 dB)  s=0.04379
  固定 s* + STE    = 1.6901e-02 (+4.80 dB)
  离线网格最优 s*（不训练）= 1.7522e-02 (+4.64 dB)  s=0.0

[save] /Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/lsq_learned_step_size/results/lsq_end_to_end_comparison.png


'/Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/lsq_learned_step_size/results/lsq_end_to_end_comparison.png'

## 实验 F：per-tensor vs per-channel LSQ

18 篇 §8 提到 LLM 场景里 LSQ 通常配 per-channel / per-group。这里补一个**粒度 × 可学习性**的
正交性检验：把 $s$ 从标量换成逐输出通道向量（$m=64$ 个 scale），并把"粒度"和"可学习"拆成
可分别归因的两步（六个 arm）。注意 per-channel 时 $g_j=1/\sqrt{n\,Q_P}$（$n=128$ 是该通道的
元素数），梯度按通道求和——**$n_Q$ 从 8192 降到 128，$g$ 反而变大 8 倍**，这是容易踩的实现细节。

In [8]:
def train_lsq_perchannel(W0, s0_vec, b, steps, lr, mom=0.9, learn_s=True, train_w=True):
    """逐输出通道 scale：s 形状 (m, 1)，g_j = 1/sqrt(n * Q_P)，梯度按通道求和。"""
    m, n = W0.shape
    gvec = 1.0 / np.sqrt(n * QMAX(b))
    W = W0.copy(); s = np.asarray(s0_vec, float).reshape(m, 1).copy()
    vw = np.zeros_like(W); vs = np.zeros_like(s)
    for _ in range(steps):
        _, gWq = loss_and_grad(X, Y, fq(W, s, b))
        if train_w:
            vw = mom * vw + gWq * ste_mask(W, s, b)
            W = W - lr * vw
        if learn_s:
            gs = np.sum(gWq * dhat_ds(W, s, b), axis=1, keepdims=True)
            vs = mom * vs + gvec * gs
            s = np.maximum(s - lr * vs, 1e-9)
    return W, s, loss_and_grad(X, Y, fq(W, s, b))[0]


b = B_MAIN
s_mm_pc = np.max(np.abs(W0), axis=1, keepdims=True) / QMAX(b)
s_lsq_pc = 2.0 * np.mean(np.abs(W0), axis=1, keepdims=True) / np.sqrt(QMAX(b))

rows_F = [dict(arm="per-tensor min-max PTQ", n_scale=1, loss=L_ptq_mm, lr=0.0),
          dict(arm="per-channel min-max PTQ", n_scale=W0.shape[0],
               loss=loss_and_grad(X, Y, fq(W0, s_mm_pc, b))[0], lr=0.0)]
best_pc_s_only = None
for lr in CFG["lr_grid"]:
    # per-channel：只训 W（s 固定 min-max）
    _, _, L = train_lsq_perchannel(W0, s_mm_pc, b, steps=steps, lr=lr, learn_s=False, train_w=True)
    rows_F.append(dict(arm="per-channel min-max + STE", n_scale=W0.shape[0], loss=L, lr=lr))
    # per-channel LSQ：只学 s
    _, s_e, L = train_lsq_perchannel(W0, s_lsq_pc, b, steps=steps, lr=lr, learn_s=True, train_w=False)
    rows_F.append(dict(arm="per-channel LSQ (s only)", n_scale=W0.shape[0], loss=L, lr=lr))
    if best_pc_s_only is None or L < best_pc_s_only[1]:
        best_pc_s_only = (s_e, L)
    # per-channel LSQ：s + W
    _, s_e, L = train_lsq_perchannel(W0, s_lsq_pc, b, steps=steps, lr=lr, learn_s=True, train_w=True)
    rows_F.append(dict(arm="per-channel LSQ (s + W)", n_scale=W0.shape[0], loss=L, lr=lr))
    # per-tensor LSQ 对照
    _, _, L, _ = train_lsq(W0, s_lsq_all[b], b, steps=steps, lr=lr, g=g_lsq,
                           learn_s=True, train_w=True)
    rows_F.append(dict(arm="per-tensor LSQ (s + W)", n_scale=1, loss=L, lr=lr))

best_F = {}
for r in rows_F:
    if r["arm"] not in best_F or r["loss"] < best_F[r["arm"]]["loss"]:
        best_F[r["arm"]] = r

log("=" * 78)
log(f"[F] 粒度 × 可学习性（bit={b}, steps={steps}，各 arm 取 lr 网格中的最好）")
log(f"{'arm':>28} {'#scale':>7} {'最好 lr':>9} {'损失':>14} {'相对 per-tensor min-max':>12}")
for name in ["per-tensor min-max PTQ", "per-channel min-max PTQ", "per-tensor LSQ (s + W)",
             "per-channel min-max + STE", "per-channel LSQ (s only)", "per-channel LSQ (s + W)"]:
    r = best_F[name]
    log(f"{r['arm']:>28} {r['n_scale']:>7} {r['lr']:>9.0e} {r['loss']:>14.4e} "
        f"{10*np.log10(L_ptq_mm/r['loss']):>+11.2f}dB")
log("-" * 78)
L_pc_ptq = best_F["per-channel min-max PTQ"]["loss"]
L_pc_lsq_s = best_F["per-channel LSQ (s only)"]["loss"]
L_pc_lsq = best_F["per-channel LSQ (s + W)"]["loss"]
L_ts_lsq = best_F["per-tensor LSQ (s + W)"]["loss"]
log(f"  粒度收益（都在 min-max 下）= {10*np.log10(L_ptq_mm/L_pc_ptq):+.2f} dB")
log(f"  可学习收益（per-channel 只学 s）= {10*np.log10(L_pc_ptq/L_pc_lsq_s):+.2f} dB")
log(f"  再叠加 W 训练               = {10*np.log10(L_pc_lsq_s/L_pc_lsq):+.2f} dB")
log(f"  per-channel：min-max -> LSQ 总计 = {10*np.log10(L_ptq_mm/L_pc_lsq):+.2f} dB")
log(f"  per-tensor -> per-channel（都学）= {10*np.log10(L_ts_lsq/L_pc_lsq):+.2f} dB")

fig, ax = plt.subplots(1, 2, figsize=(11.5, 4.2))
names = ["per-tensor min-max PTQ", "per-channel min-max PTQ", "per-tensor LSQ (s + W)",
         "per-channel min-max + STE", "per-channel LSQ (s only)", "per-channel LSQ (s + W)"]
bars = ax[0].bar(range(len(names)), [best_F[n]["loss"] for n in names],
                 color=["#C44E52", "#DD8452", "#55A868", "#937860", "#8172B3", "#4C72B0"])
ax[0].set_yscale("log"); ax[0].set_ylabel("task loss")
ax[0].set_xticks(range(len(names)))
ax[0].set_xticklabels(names, fontsize=7.5, rotation=16, ha="right")
ax[0].set_title("[F] Granularity vs learnability: which gain is real here?")
for i, n in enumerate(names):
    v = best_F[n]["loss"]
    ax[0].text(i, v, f"{10*np.log10(L_ptq_mm/v):+.1f}dB", ha="center", va="bottom", fontsize=7.5)

s_ratio = (best_pc_s_only[0] / s_mm_pc).ravel()
ax[1].hist(s_ratio, bins=30, color="#4C72B0", alpha=0.85)
ax[1].axvline(1.0, color="#C44E52", ls="--", label="min-max (ratio=1)")
ax[1].axvline(float(np.median(s_ratio)), color="#55A868", ls=":", lw=1.5,
              label=f"median={np.median(s_ratio):.2f}")
ax[1].set_xlabel("learned s / s_minmax (per output channel)")
ax[1].set_ylabel("#channels")
ax[1].set_title("[F] Learned per-channel scales sit below min-max")
ax[1].legend(fontsize=8)
savefig(fig, "lsq_per_channel_vs_per_tensor.png")

[F] 粒度 × 可学习性（bit=4, steps=400，各 arm 取 lr 网格中的最好）
                         arm  #scale     最好 lr             损失 相对 per-tensor min-max
      per-tensor min-max PTQ       1     0e+00     5.1021e-02       +0.00dB
     per-channel min-max PTQ      64     0e+00     1.5430e-02       +5.19dB
      per-tensor LSQ (s + W)       1     1e-03     1.8280e-02       +4.46dB
   per-channel min-max + STE      64     1e-03     1.5150e-02       +5.27dB
    per-channel LSQ (s only)      64     1e-02     1.4516e-02       +5.46dB
     per-channel LSQ (s + W)      64     1e-02     1.7636e-02       +4.61dB
------------------------------------------------------------------------------
  粒度收益（都在 min-max 下）= +5.19 dB
  可学习收益（per-channel 只学 s）= +0.27 dB
  再叠加 W 训练               = -0.85 dB
  per-channel：min-max -> LSQ 总计 = +4.61 dB
  per-tensor -> per-channel（都学）= +0.16 dB


[save] /Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/lsq_learned_step_size/results/lsq_per_channel_vs_per_tensor.png


'/Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/lsq_learned_step_size/results/lsq_per_channel_vs_per_tensor.png'

## 结论汇总

把六个实验的关键数字收在一处，同时写入 `results/results.json` 与 `results/stdout.txt`。

In [9]:
summary = {
    "meta": {"mode": MODE, "cfg": {k: (list(v) if isinstance(v, tuple) else v) for k, v in CFG.items()},
             "numpy": np.__version__, "seed": SEED,
             "task": "overdetermined linear regression (18 篇 §2 task A)",
             "W_shape": list(W0.shape)},
    "A_scale_sweep": rows_A,
    "B_gradient_decomposition": rows_B,
    "B_zero_crossing": zero_cross,
    "C_self_stabilization": rows_C,
    "D_imbalance": rows_D,
    "D_g_ablation": rows_G,
    "E_end_to_end": rows_E,
    "F_granularity": [{k: (float(v) if isinstance(v, float) else v) for k, v in r.items()} for r in rows_F],
}

log("")
log("=" * 78)
log("结论汇总")
log("=" * 78)
r4 = rows_A[[r["bit"] for r in rows_A].index(4)]
log("1) [A] 4-bit：min-max %.4e -> MSE 最优 %.4e，白捡 %.2f dB（LSQ 的收益天花板）"
    % (r4["L_minmax"], r4["L_opt"], r4["gain_db"]))
log("2) [B] 4-bit：两项过零点 s_zero=%.5f 与网格最优 s*=%.5f 只差 %.2f%%，"
    "过零点处残差占 %.3f%%；逐元素不对称 %.0fx"
    % (zero_cross["s_zero"], zero_cross["s_grid_opt"], zero_cross["rel_dev_pct"],
       zero_cross["resid_pct"], [r for r in rows_B if r["bit"] == 4][0]["asym"]))
cur_c = [r for r in rows_C if r["lr"] == lr_trace]
log("3) [C] 大 lr(%.0e) 下 s0 从 %.1fx 到 %.1fx：收敛 s 离散 %.1f%%；"
    "小 lr(%.0e) 下 10x 初值直接崩到 %.3e"
    % (lr_trace, CFG["s0_mult"][0], CFG["s0_mult"][-1],
       100 * (max(r["s_end"] for r in cur_c) / min(r["s_end"] for r in cur_c) - 1),
       min(lr_c_list), max(r["loss"] for r in rows_C if r["lr"] == min(lr_c_list))))
d4 = [r for r in rows_D if r["bit"] == 4][0]
log("4) [D] 4-bit 实测 R=%.0f（论文估计 %.0f，低估 %.1fx）；相干和/随机和 = %.1fx"
    % (d4["R"], d4["pred"], d4["ratio"], d4["coherent"]))
log("5) [E] LSQ(s+W) 最好 %.4e（%+.2f dB）；只学 s %.4e（%+.2f dB）；"
    "相对离线网格最优固定 s 差 %.3f dB"
    % (best_lsq["loss"], best_lsq["db"], best_sonly["loss"], best_sonly["db"],
       10 * np.log10(L_opt_ptq / best_sonly["loss"])))
log("6) [F] per-channel LSQ(s+W) %.4e（%+.2f dB）：其中粒度 %+.2f dB、"
    "per-channel 只学 s %+.2f dB、再训 W %+.2f dB"
    % (L_pc_lsq, 10 * np.log10(L_ptq_mm / L_pc_lsq), 10 * np.log10(L_ptq_mm / L_pc_ptq),
       10 * np.log10(L_pc_ptq / L_pc_lsq_s), 10 * np.log10(L_pc_lsq_s / L_pc_lsq)))
log("=" * 78)
log("工程 takeaway：s0 = 2<|v|>/sqrt(Q_P)（宁小勿大）；g = 1/sqrt(n_Q*Q_P)（SGD 必加，Adam 无害）；")
log("                s 放 weight_decay=0 的 param group；用硬量化前向做最终评估。")

with open(os.path.join(RES, "results.json"), "w") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False, default=float)
with open(os.path.join(RES, "stdout.txt"), "w") as f:
    f.write("\n".join(_LINES) + "\n")
log(f"[save] {os.path.join(RES, 'results.json')}")
log(f"[save] {os.path.join(RES, 'stdout.txt')}")


结论汇总
1) [A] 4-bit：min-max 5.1021e-02 -> MSE 最优 1.7522e-02，白捡 4.64 dB（LSQ 的收益天花板）
2) [B] 4-bit：两项过零点 s_zero=0.04378 与网格最优 s*=0.04149 只差 5.51%，过零点处残差占 0.000%；逐元素不对称 240x
3) [C] 大 lr(3e-02) 下 s0 从 0.1x 到 10.0x：收敛 s 离散 7.6%；小 lr(1e-03) 下 10x 初值直接崩到 5.224e-01
4) [D] 4-bit 实测 R=2167（论文估计 239，低估 9.0x）；相干和/随机和 = 32.2x
5) [E] LSQ(s+W) 最好 1.8280e-02（+4.46 dB）；只学 s 1.7495e-02（+4.65 dB）；相对离线网格最优固定 s 差 0.007 dB
6) [F] per-channel LSQ(s+W) 1.7636e-02（+4.61 dB）：其中粒度 +5.19 dB、per-channel 只学 s +0.27 dB、再训 W -0.85 dB
工程 takeaway：s0 = 2<|v|>/sqrt(Q_P)（宁小勿大）；g = 1/sqrt(n_Q*Q_P)（SGD 必加，Adam 无害）；
                s 放 weight_decay=0 的 param group；用硬量化前向做最终评估。
[save] /Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/lsq_learned_step_size/results/results.json
[save] /Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/lsq_learned_step_size/results/stdout.txt


## 下一步（待确认后展开）

本目录只覆盖 **LSQ** 一个算法。QAT 侧还有这些可以照此模板补：

| 目录（拟） | 算法 | 核心要验证的一句话 |
|---|---|---|
| `pact_learnable_clip/` | PACT | 可学习 clip 上界；α 动力学比权重慢 1–2 个数量级；λ 极难标定 |
| `dsq_soft_quant/` | DSQ | 训练前向 ≠ 部署前向，train→deploy gap 33–43 dB；max 偏差恒为 Δ/2 |
| `adaround_brecq_qdrop/` | AdaRound / BRECQ / QDrop | 舍入方向本身是优化变量；PTQ→QAT 的桥 |
| `fake_quant_ste/` | 伪量化 + STE（17 篇） | 网格吸附效应、STE 的有偏性 |
| `qat_distillation/` | 蒸馏式 QAT（20 篇） | 用 FP teacher 带低比特 student |

确认内容没问题后：把 `MODE` 改成 `"full"` 重跑本 notebook 即为最终版，再按上表批量补齐。